# Orders data Stage Load

In [11]:
# Configuration variables for the data pipeline
source_dir = "Volumes/demo-external-catalog/default/incremental_load/orders_data/source/"
archived_dir = "Volumes/demo-external-catalog/default/incremental_load/orders_data/archive/"
stage_table = "demo_external_catalog.default.orders_stage"
error_table = "demo_external_catalog.default.orders_errors"
# the stage table here is wherre the ingested or extracted data is temporarily stored before it is transformed and loaded into the final destination table. the error table is where any errors that occur during the ingestion or transformation process are logged for later review and debugging.

print(f"Processing orders data from: {source_dir}")
print(f"Archiving processed files to: {archived_dir}")
print(f"Staging table: {stage_table}")
print(f"Error table: {error_table}")

Processing orders data from: Volumes/demo-external-catalog/default/incremental_load/orders_data/source/
Archiving processed files to: Volumes/demo-external-catalog/default/incremental_load/orders_data/archive/
Staging table: demo_external_catalog.default.orders_stage
Error table: demo_external_catalog.default.orders_errors


In [12]:
# Import required libraries
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import json

# Define the schema for the orders data

orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("order_date", DateType(), False),
    StructField("order_amount", DecimalType(10, 2), False),
    StructField("currency", StringType(), False),
    StructField("payment_method", StringType(), False),
    StructField("shipping_address", StringType(), False),
    StructField("order_status", StringType(), False),
    StructField("created_timestamp", TimestampType(), False)
])

print("schema defined for orders data")

schema defined for orders data


In [ ]:
# Read and vaidate orders data
try:
    # Read CSV files with schema validation
    df_orders = spark.read.schema(orders_schema).csv(source_dir, header = True, dateFormat = "yyyy-MM-dd", timestampFormat = "yyyy-MM-dd HH:mm:ss")

    # Add processing metadata 
    df_orders = df_orders.withColumn("processed_timestamp", F.current_timestamp())\
        .withColumn("batch_id", F.lit(f"batch_{datetime.now().strftime('%Y%m%d%H%M%S')}"))\
        .withColumn("source_system", F.lit("ecommerce_orders"))
    
    # the code above adds three new columns to the DataFrame:
    # 1. processed_timestamp: This column captures the current timestamp when the data is being processed.
    # 2. batch_id: This column generates a unique identifier for the batch of data being processed, using the current date and time.
    # 3. source_system: This column indicates the source of the data, which in this case is labeled as "ecommerce_orders".
    
    # Data quality checks
    total_records = df_orders.count()
    null_order_ids = df_orders.filter(F.col("order_id").isNull()).count()
    null_customer_ids = df_orders.filter(F.col("customer_id").isNull()).count()
    invalid_amounts = df_orders.filter(F.col("order_amount").isNull() | F.col("order_amount") <= 0).count()

    print (f"Total records processed: {total_records}") 
    print (f"Records with null order_id: {null_order_ids}")
    print (f"Records with null customer_id: {null_customer_ids}") 
    print (f"Records with Invalid amounts: {invalid_amounts}")

    # Filter out valid records - Fixed boolean logic
    df_valid_orders = df_orders.filter(
        F.col("order_id").isNotNull() &
        F.col("customer_id").isNotNull() &
        (F.col("order_amount") > 0)
    )

    # capture invaid records or error handling - fixed boolean logic
    df_invalid_orders = df_orders.filter(
        F.col("order_id").isNull() |
        F.col("customer_id").isNull() |
        (F.col("order_amount") <= 0)
    )

    valid_records = df_valid_orders.count()
    invalid_records = df_invalid_orders.count()

    print(f"Valid records to be ingested: {valid_records}")
    print(f"Invalid records to be logged: {invalid_records}")
except Exception as e:
    print(f"Error reading orders data: {str(e)}")
    raise


In [ ]:
# write valid data to staging table
try:
    # Create or overwrite staging table
    df_valid_orders.write.format("delta").mode("overwrite").saveAsTable(stage_table)
    print(f"Successfully loaded {valid_records} valid orders to the staging table")

    # write invalid records to error table for investigation
    if invalid_records > 0:
        df_invalid_orders.withColumn("error_reason", F.lit("Data quality validation failed"))\
            .withColumn("error_timestamp", F.current_timestamp())\
            .write.format("delta").mode("append").saveAsTable(error_table)
        print(f"Logged {invalid_records} invalid records to the error table")
except Exception as e:
    print(f"Error writing to tables: {str(e)}")
    raise

In [ ]:
# Archive processed files
try:
    # List all files in the source directory
    files = dbutils.fs.ls(source_dir)

    archived_files_count = 0
    for file in files:
        if file.name.endswith(".csv"):
            src_path = file.path
            archive_path = archived_dir + file.name

            # Move the file to the archived directory
            dbutils.fs.mv(src_path, archive_path)
            archived_files_count += 1

    print(f"Successfully archived {archived_files_count} files")
except Exception as e:
    print(f"Error archiving files: {str(e)}")
    raise

In [ ]:
# Log processing Summary
processing_summary = {
    "task": "orders_stage_load",
    "Timestamp": datetime.now().isoformat(),
    "total_records": total_records,
    "valid_records": valid_records,
    "invalid_records": invalid_records,
    "archived_files": archived_files_count,
    "status": "Success" if invalid_records == 0 else "Completed with errors"
}
# saving summary stats as a dictionary, which will be printed and also stored in a table for monitoring and auditing purposes.

print("Processing Summary:")
print(json.dumps (processing_summary, indent=2))
# prints the dictionary as a neatly formatted JSON string, making it easier to read and understand the processing summary.

# store summary in a table for monitoring and auditing purposes
summary_df = spark.createDataFrame([processing_summary])
summary_df.write.format("delta").mode("append").saveAsTable("demo_external_catalog.default.processing_summary")
# the code above creates a DataFrame from the processing_summary dictionary and appends it to a Delta table named "processing_summary" in the "demo_external_catalog.default" schema. This allows for easy tracking and auditing of the data processing tasks over time.